# 01 – Priprava mesečnega nabora turističnih podatkov

Tri vire združimo v eno urejeno tabelo, kjer je ena vrstica **vir × obseg nastanitev × občina × mesec**:

| Vir | Država | Obseg (`scope`) | Obdobje | Mere |
|---|---|---|---|---|
| ASTAT | Italija (Južna Tirolska) | `hotel` in `all` | 2005 → | prihodi, prenočitve, postelje |
| ISTAT | Italija (vse regije) | `all` | 2022 → 2025 | prihodi, prenočitve |
| BFS | Švica | `hotel` | 2013 → | prihodi, prenočitve |

Rezultat: `data/processed/tourism_monthly.csv`. Znane pasti v podatkih so opisane v `PLAN.md` in `CLAUDE.md`.

In [1]:
from pathlib import Path
import pandas as pd
import requests

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# notebooks/ lives one level below the project root
ROOT = Path.cwd() if (Path.cwd() / "astat_tourism_monthly").exists() else Path.cwd().parent
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
ROOT

WindowsPath('C:/Users/lenar/Documents/Šola/FERI/3. Letnik/MOS/Projekt')

## Nadmorske višine

In [2]:
elev = pd.read_csv(ROOT / "data/reference/elevation.csv", dtype={"muni_id": str})
# several values per municipality (e.g. village vs. highest point): use the lowest, flag a large spread
elev["elev_m"] = elev["elev_min_m"]
elev["elev_spread_m"] = elev["elev_max_m"] - elev["elev_min_m"]
elev["elev_ambiguous"] = elev["elev_spread_m"] > 100
elev = elev[["country", "muni_id", "name", "elev_m", "elev_spread_m", "elev_ambiguous"]]
print(elev.groupby("country").agg(n=("muni_id", "size"), ambiguous=("elev_ambiguous", "sum")))
elev.head()

            n  ambiguous
country                 
CH       3266        315
IT       7992          0


,country,muni_id,name,elev_m,elev_spread_m,elev_ambiguous
0,IT,001001,Agliè,315.0,0.0,False
1,IT,001002,Airasca,257.0,0.0,False
2,IT,001003,Ala di Stura,1080.0,0.0,False
3,IT,001004,Albiano d'Ivrea,230.0,0.0,False
4,IT,001006,Almese,364.0,0.0,False


## ASTAT – Južna Tirolska

Stolpec `TOUR_GEO` meša občine (`021xxx`) z vsotami za okraje in provinco (`DIST*`, `ITH10`), ki jih izločimo.
`OBS_VALUE = ".."` pomeni zaupen ali manjkajoč podatek in postane `NaN`, ne nič.

In [3]:
files = sorted((ROOT / "astat_tourism_monthly").glob("*.csv"))
raw = pd.concat(
    (pd.read_csv(f, sep=";", dtype=str,
                 usecols=["TOUR_GEO", "CATEGORY", "INDICATOR", "TIME_PERIOD", "OBS_VALUE"])
     for f in files),
    ignore_index=True,
)
print(f"{len(files)} files, {len(raw):,} rows")

raw = raw[raw["TOUR_GEO"].str.startswith("021")]                      # municipalities only
raw = raw[raw["CATEGORY"].isin(["A", "TOTAL"]) &
          raw["INDICATOR"].isin(["ARRIVALS_N", "OVERNIGHTS_N", "BEDS_N"])]
raw["OBS_VALUE"] = pd.to_numeric(raw["OBS_VALUE"], errors="coerce")  # ".." -> NaN

astat = (raw.pivot_table(index=["TOUR_GEO", "TIME_PERIOD", "CATEGORY"],
                         columns="INDICATOR", values="OBS_VALUE", aggfunc="first", dropna=False)
            .reset_index())
astat.columns.name = None
astat = astat.rename(columns={"TOUR_GEO": "muni_id", "ARRIVALS_N": "arrivals",
                              "OVERNIGHTS_N": "overnights", "BEDS_N": "beds"})
astat["scope"] = astat.pop("CATEGORY").map({"A": "hotel", "TOTAL": "all"})
astat["date"] = pd.to_datetime(astat.pop("TIME_PERIOD") + "-01")
astat["country"], astat["source"] = "IT", "ASTAT"
astat["region"] = "BOLZANO - BOZEN"
print(f"{len(astat):,} rows, {astat['muni_id'].nunique()} municipalities, "
      f"{astat['date'].min():%Y-%m} to {astat['date'].max():%Y-%m}")
astat.head()

261 files, 1,220,870 rows


60,552 rows, 116 municipalities, 2005-01 to 2026-09


,muni_id,arrivals,beds,overnights,scope,date,country,source,region
0,021001,963.0,671.0,7407.0,hotel,2005-01-01,IT,ASTAT,BOLZANO - BOZEN
1,021001,1060.0,1003.0,8050.0,all,2005-01-01,IT,ASTAT,BOLZANO - BOZEN
2,021001,1083.0,671.0,5889.0,hotel,2005-02-01,IT,ASTAT,BOLZANO - BOZEN
3,021001,1143.0,1003.0,6060.0,all,2005-02-01,IT,ASTAT,BOLZANO - BOZEN
4,021001,916.0,671.0,5777.0,hotel,2005-03-01,IT,ASTAT,BOLZANO - BOZEN


## ISTAT – vsa Italija

Mesečni podatki po občinah, 2022–2025, vse vrste nastanitev (brez ločitve na hotele). Občine z zaupnimi podatki manjkajo.
Južna Tirolska je tu prav tako zajeta, zato se prekriva z ASTAT; vira ločimo s stolpcem `source` in ju spodaj primerjamo.

In [4]:
im = pd.read_csv(ROOT / "istat_tourism/monthly_municipal.csv", dtype={"istat_code": str})
istat = (im.pivot_table(index=["istat_code", "municipality", "region", "year", "month"],
                        columns="indicator", values="value", aggfunc="first")
           .reset_index())
istat.columns.name = None
istat = istat.rename(columns={"istat_code": "muni_id", "municipality": "muni_name",
                              "nights": "overnights"})
istat["date"] = pd.to_datetime(dict(year=istat.pop("year"), month=istat.pop("month"), day=1))
istat["country"], istat["source"], istat["scope"] = "IT", "ISTAT", "all"
print(f"{len(istat):,} rows, {istat['muni_id'].nunique()} municipalities, "
      f"{istat['date'].min():%Y-%m} to {istat['date'].max():%Y-%m}")
istat.head()

118,512 rows, 2469 municipalities, 2022-01 to 2025-12


,muni_id,muni_name,region,arrivals,overnights,date,country,source,scope
0,001003,Ala di Stura,PIEMONTE,128,299,2022-01-01,IT,ISTAT,all
1,001003,Ala di Stura,PIEMONTE,195,307,2022-02-01,IT,ISTAT,all
2,001003,Ala di Stura,PIEMONTE,93,172,2022-03-01,IT,ISTAT,all
3,001003,Ala di Stura,PIEMONTE,148,310,2022-04-01,IT,ISTAT,all
4,001003,Ala di Stura,PIEMONTE,172,368,2022-05-01,IT,ISTAT,all


## BFS – Švica

Samo hotelski sektor, 186 večjih turističnih občin. CSV-ji vsebujejo **imena** občin, zato jih prek metapodatkov tabele preslikamo v uradne številke občin BFS (shranjene v `data/reference/bfs_communes.csv`). Mesec preberemo iz imena datoteke. Uporabimo le stolpce s *skupnim* številom gostov iz vseh držav.

In [5]:
communes_path = ROOT / "data/reference/bfs_communes.csv"
if not communes_path.exists():
    meta = requests.get("https://www.pxweb.bfs.admin.ch/api/v1/en/"
                        "px-x-1003020000_101/px-x-1003020000_101.px", timeout=60).json()
    gem = next(v for v in meta["variables"] if v["code"] == "Gemeinde")
    pd.DataFrame({"muni_id": gem["values"], "muni_name": gem["valueTexts"]}).to_csv(
        communes_path, index=False, encoding="utf-8")
communes = pd.read_csv(communes_path, dtype=str)
assert communes["muni_name"].is_unique

TOTAL = "Visitors' country of residence - total"
parts = []
for f in sorted((ROOT / "bfs_tourism_monthly").glob("*.csv")):
    d = pd.read_csv(f, encoding="utf-8",
                    usecols=["Commune", f"{TOTAL} Arrivals", f"{TOTAL} Overnight stays"])
    d.columns = ["muni_name", "arrivals", "overnights"]
    d["date"] = pd.to_datetime(f.stem + "-01")
    parts.append(d)
bfs = pd.concat(parts, ignore_index=True)
for c in ["arrivals", "overnights"]:
    bfs[c] = pd.to_numeric(bfs[c], errors="coerce")        # "..." -> NaN

bfs = bfs.merge(communes, on="muni_name", how="left", validate="m:1")
assert bfs["muni_id"].notna().all(), "unmapped commune names"
bfs["country"], bfs["source"], bfs["scope"] = "CH", "BFS", "hotel"
print(f"{len(bfs):,} rows, {bfs['muni_id'].nunique()} communes, "
      f"{bfs['date'].min():%Y-%m} to {bfs['date'].max():%Y-%m}")
bfs.head()

30,504 rows, 186 communes, 2013-01 to 2026-08


,muni_name,arrivals,overnights,date,muni_id,country,source,scope
0,Kloten,12826.0,16822.0,2013-01-01,62,CH,BFS,hotel
1,Opfikon,21550.0,31788.0,2013-01-01,66,CH,BFS,hotel
2,Dübendorf,1601.0,3379.0,2013-01-01,191,CH,BFS,hotel
3,Uster,909.0,2608.0,2013-01-01,198,CH,BFS,hotel
4,Winterthur,5702.0,11487.0,2013-01-01,230,CH,BFS,hotel


## Združevanje in dodajanje nadmorske višine

In [6]:
cols = ["country", "source", "scope", "muni_id", "muni_name", "region", "date",
        "arrivals", "overnights", "beds"]
df = pd.concat([astat, istat, bfs], ignore_index=True)
df = df.reindex(columns=cols)

# names: ASTAT has none, take them from the elevation table (Wikidata labels), else the code
names = elev.drop_duplicates(["country", "muni_id"]).set_index(["country", "muni_id"])["name"]
lookup = pd.MultiIndex.from_frame(df[["country", "muni_id"]])
df["muni_name"] = df["muni_name"].fillna(pd.Series(names.reindex(lookup).to_numpy(), index=df.index))
df["muni_name"] = df["muni_name"].fillna(df["muni_id"])

df = df.merge(elev.drop(columns="name"), on=["country", "muni_id"], how="left", validate="m:1")
df["year"], df["month"] = df["date"].dt.year, df["date"].dt.month
df = df.sort_values(["country", "source", "scope", "muni_id", "date"], ignore_index=True)
df.to_csv(OUT / "tourism_monthly.csv", index=False, encoding="utf-8")
print(f"{len(df):,} rows -> {OUT / 'tourism_monthly.csv'}")
df.sample(5, random_state=1)

209,568 rows -> C:\Users\lenar\Documents\Šola\FERI\3. Letnik\MOS\Projekt\data\processed\tourism_monthly.csv


,country,source,scope,muni_id,muni_name,region,date,arrivals,overnights,beds,elev_m,elev_spread_m,elev_ambiguous,year,month
155776,IT,ISTAT,all,041009,Carpegna,MARCHE,2023-05-01,1127.0,2263.0,NaN,748.0,0.0,False,2023,5
61853,IT,ASTAT,hotel,021005,Avelengo,BOLZANO - BOZEN,2007-06-01,2866.0,16653.0,649.0,1290.0,0.0,False,2007,6
106876,IT,ISTAT,all,011030,Vernazza,LIGURIA,2024-05-01,6619.0,15800.0,NaN,3.0,0.0,False,2024,5
85461,IT,ASTAT,hotel,021097,Terlano,BOLZANO - BOZEN,2017-04-01,2146.0,6685.0,403.0,250.0,0.0,False,2017,4
20504,CH,BFS,hotel,571,Beatenberg,NaN,2013-05-01,3562.0,8435.0,NaN,1129.0,549.0,True,2013,5


## Preverjanje

In [7]:
summary = (df.groupby(["country", "source", "scope"])
             .agg(municipalities=("muni_id", "nunique"), rows=("muni_id", "size"),
                  first=("date", "min"), last=("date", "max"),
                  missing_overnights=("overnights", lambda s: s.isna().mean()),
                  no_elevation=("elev_m", lambda s: s.isna().mean())))
summary["first"] = summary["first"].dt.strftime("%Y-%m")
summary["last"] = summary["last"].dt.strftime("%Y-%m")
summary.round(3)

municipalities    rows    first     last  missing_overnights  no_elevation
country source scope                                                                            
CH      BFS    hotel             186   30504  2013-01  2026-08               0.037         0.032
IT      ASTAT  all               116   30276  2005-01  2026-09               0.019         0.000
               hotel             116   30276  2005-01  2026-09               0.061         0.000
        ISTAT  all              2469  118512  2022-01  2025-12               0.000         0.015

In [8]:
# Which municipalities lack an elevation?
no_elev = df[df["elev_m"].isna()].drop_duplicates(["country", "muni_id"])
print(len(no_elev), "municipalities without elevation")
no_elev[["country", "source", "muni_id", "muni_name"]].head(15)

42 municipalities without elevation


,country,source,muni_id,muni_name
22304,CH,BFS,5892,Blonay - Saint-Légier
23452,CH,BFS,6037,Val de Bagnes
23616,CH,BFS,6077,Goms
26568,CH,BFS,6513,Laténa
26896,CH,BFS,6621,Genève
28372,CH,BFS,700,Moutier (BE)
108192,IT,ISTAT,012140,Vizzola Ticino
119232,IT,ISTAT,020061,Sermide e Felonica
129216,IT,ISTAT,022242,Contà
129264,IT,ISTAT,022243,Madruzzo


### Primerjava ASTAT in ISTAT

Iste občine Južne Tirolske, dva izdajatelja, 2022–2025, vse nastanitve. Razmerje prenočitev bi moralo biti blizu 1.

In [9]:
a = df[(df.source == "ASTAT") & (df.scope == "all")][["muni_id", "date", "overnights"]]
i = df[(df.source == "ISTAT") & (df.region == "BOLZANO - BOZEN")][["muni_id", "date", "overnights"]]
cmp = a.merge(i, on=["muni_id", "date"], suffixes=("_astat", "_istat")).dropna()
cmp["ratio"] = cmp["overnights_istat"] / cmp["overnights_astat"]
print(f"{len(cmp):,} matched municipality-months, {cmp['muni_id'].nunique()} municipalities")
print("ratio ISTAT/ASTAT quantiles:")
print(cmp["ratio"].quantile([.05, .25, .5, .75, .95]).round(3).to_string())
print("correlation:", round(cmp[["overnights_astat", "overnights_istat"]].corr().iloc[0, 1], 4))

5,376 matched municipality-months, 112 municipalities
ratio ISTAT/ASTAT quantiles:
0.05    0.998
0.25    1.000
0.50    1.000
0.75    1.000
0.95    1.000
correlation: 1.0
